# Task 2: LoRA Amharic video with Wan2.1-1.3B (Colab T4)
Runtime > Change runtime type > T4 GPU. Run the cells in order.

**1. Get the code.** Clones the repo with your `GH_TOKEN` secret (GitHub token, starts with `ghp_`). If that fails it asks you to upload `wan.zip`.

In [ ]:
import os, glob
from google.colab import userdata
REPO = "/content/repo"
if not os.path.exists(REPO):
    try:
        tok = userdata.get("GH_TOKEN").strip()
        assert tok.startswith(("ghp_", "github_pat_")), "GH_TOKEN must be a GitHub token"
        !git clone https://{tok}@github.com/simachewalamneh/video_gen.git {REPO}
    except Exception as e:
        print("Clone failed:", e, "\nUpload wan.zip instead.")
        from google.colab import files
        files.upload()
        z = glob.glob("/content/wan*.zip")[0]
        !mkdir -p {REPO} && unzip -oq "{z}" -d {REPO}
WORK = os.path.dirname(glob.glob(f"{REPO}/**/train_lora.py", recursive=True)[0])
%cd {WORK}

**2. Install.** Ge'ez font, packages, removes the incompatible `torchao`, applies the `peft` import fix (safe to repeat) and deletes any stale caches.

In [ ]:
!apt-get -q install -y fonts-noto-core > /dev/null
!pip -q install -r requirements.txt
!pip -q uninstall -y torchao
!sed -i 's/from diffusers.utils import get_peft_model_state_dict/from peft.utils import get_peft_model_state_dict/' train_lora.py
!rm -f outputs/*/cache*.pt
!grep -n "peft.utils" train_lora.py

**3. Synthetic Amharic (Ge'ez) dataset.** 40 short clips of animated Amharic words with captions. Or put your own mp4s plus `metadata.jsonl` in `data/amharic_synth`.

In [ ]:
!python make_amharic_data.py --n 40

**4. Train the LoRA.** Your earlier run: 200 steps took 674 s at a 3.2 GB peak. Set `STEPS = 10` for a quick smoke test. The first run also builds the cache (text encoder + VAE), which takes a few minutes.

In [ ]:
STEPS = 400
!python train_lora.py --steps {STEPS} --accum 2 --out outputs/lora_r16

**5. Loss curve.** The raw loss is noisy (one sample and one random timestep per step), so look at the moving average.

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
d = pd.read_csv("outputs/lora_r16/train_log.csv"); d["avg"] = d.loss.rolling(20, min_periods=1).mean()
plt.plot(d.step, d.loss, alpha=.3, label="raw"); plt.plot(d.step, d.avg, label="moving average (20)")
plt.xlabel("step"); plt.ylabel("flow-matching MSE"); plt.legend(); plt.show()

**6. Generate videos** from Amharic prompts: base model first, then base + LoRA (same seeds).

In [ ]:
!python infer.py --lora_dir outputs/lora_r16 --out outputs/samples

**7. Compare.** Left of each pair is the base model, right is the LoRA.

In [ ]:
from IPython.display import Video, display
for k in range(2):
    for tag in ["base", "lora"]:
        print(tag, "prompt", k)
        display(Video(f"outputs/samples/{tag}_{k}.mp4", embed=True, width=384))

**8. Save results before the session ends.** Downloads the LoRA adapter, logs, loss graph and videos (no large caches).

In [ ]:
%cd {WORK}
!zip -r /content/task2_results.zip outputs -x "*cache*" "*.pt"
from google.colab import files
files.download("/content/task2_results.zip")